# Annular Boussinesq — reference-faithful competitive architectures

This comparison uses **F-FNO**, **CNO**, **U-Net**, and **U-NO**, with architecture choices cross-checked against the project's earlier Burgers and Cahn–Hilliard benchmark implementations.

- **F-FNO:** FourierFlow/Cahn–Hilliard-style 24-layer factorized spectral operator, shared Fourier weights and residual feed-forward blocks.
- **CNO:** Cahn–Hilliard's official-*simplified*-CNO-style encoder–decoder with filtered LeakyReLU activations, multiresolution skips and residual subnetworks, adapted for a rectangular polar grid.
- **U-Net:** four-level encoder–decoder with transpose-convolution upsampling and group normalization.
- **U-NO:** the maintained `neuralop.models.UNO` implementation, not a handwritten approximation.

**The adaptation deliberately changes the output head to K=30** so all four models learn the original AWFNO annular task: one state to the next 30 states, followed by an initial-state-only 100-step autoregressive evaluation. This is **not identical to the single-step F-FNO / direct-time CNO training protocol used for Cahn–Hilliard**; it is an architecture-faithful *task adaptation*. Polar annulus boundaries use periodic θ and nonperiodic r in local convolutions.

Read `reference_architectures_2d.py` for model details and `competitive_baselines_2d.py` for complete training/evaluation. A brief forward/backward smoke test runs **before** training; its success does not guarantee production convergence. Source is transparent and versioned in GitHub. Outputs are independently saved on Google Drive.

In [ ]:
#@title 1. Google Drive and GPU setup
from google.colab import drive
from pathlib import Path
import torch, os, sys, time, subprocess, hashlib, urllib.request
drive.mount('/content/gdrive')

DATASET = Path('/content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt')
OUTPUT_ROOT = Path('/content/gdrive/MyDrive/anus/Boussinesq_JCP')
BASELINE_ROOT = OUTPUT_ROOT / 'competitive_2d_K30'
assert DATASET.is_file(), f"Dataset missing: {DATASET}"
assert torch.cuda.is_available(), "Set Colab Runtime type to GPU first."
BASELINE_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator==2.0.0'], check=True)
from neuralop.models import UNO
print('Installed neuraloperator==2.0.0; library UNO available')
print('GPU:', torch.cuda.get_device_name(0))
print('Data:', DATASET, '| GiB:', round(DATASET.stat().st_size/1024**3, 3))
print('Baseline outputs:', BASELINE_ROOT)

In [ ]:
#@title 2. Experimental configuration
RUN_BASELINES = 'FFNO,CNO,UNet,UNO' #@param {type:'string'}
RUN_SAMPLE_SIZES = '25,50,100,200,400' #@param {type:'string'}
SEEDS = '1234' #@param {type:'string'}
EPOCHS = 150 #@param {type:'integer'}
BATCH_SIZE = 32 #@param {type:'integer'}
TEST_BATCH = 8 #@param {type:'integer'}
WIDTH = 32 #@param {type:'integer'}
SAVE_TEST_ROLLOUTS = True #@param {type:'boolean'}
FORCE_RETRAIN = False #@param {type:'boolean'}
SMOKE_TEST_FIRST = True #@param {type:'boolean'}
SHUTDOWN_AFTER_SUCCESS = True #@param {type:'boolean'}

assert set(RUN_BASELINES.split(',')).issubset({'FFNO','CNO','UNet','UNO'})
assert all(int(i) in (25,50,100,200,400) for i in RUN_SAMPLE_SIZES.split(','))
print('Models:', RUN_BASELINES)
print('N:', RUN_SAMPLE_SIZES, '| seeds:', SEEDS)
print('Reference-style F-FNO and CNO backbones; library UNO; K30 task adaptation, AdamW/cosine, fixed test split.')

In [ ]:
#@title 3. Fetch readable, versioned Python implementations from GitHub
ROOT = 'https://raw.githubusercontent.com/dimigourz/AWFNO/main/Boussinesq2D/'
SOURCES = ['annulus2d_jcp_experiments.py', 'reference_architectures_2d.py', 'competitive_baselines_2d.py']
for name in SOURCES:
    target = Path('/content') / name
    with urllib.request.urlopen(ROOT+name, timeout=90) as response:
        source = response.read()
    target.write_bytes(source)
    print(f'{name}: {len(source):,} bytes, SHA256={hashlib.sha256(source).hexdigest()}')
DRIVER = Path('/content/competitive_baselines_2d.py')
print('Human-readable source:', 'https://github.com/dimigourz/AWFNO/tree/main/Boussinesq2D')

In [ ]:
#@title 4. Smoke-test all architectures before expensive training
ENV = os.environ.copy()
ENV.update({
    'DATA_PATH': str(DATASET), 'OUTPUT_ROOT': str(OUTPUT_ROOT),
    'BASELINE_OUTPUT': str(BASELINE_ROOT),
    'BASELINES': RUN_BASELINES, 'BASELINE_SAMPLE_SIZES': RUN_SAMPLE_SIZES,
    'BASELINE_SEEDS': SEEDS, 'EPOCHS': str(EPOCHS), 'BATCH_SIZE': str(BATCH_SIZE),
    'BASELINE_EVAL_BATCH': str(TEST_BATCH), 'BASELINE_WIDTH': str(WIDTH),
    'FORCE_RETRAIN': '1' if FORCE_RETRAIN else '0',
    'SAVE_BASELINE_ROLLOUTS': '1' if SAVE_TEST_ROLLOUTS else '0',
    'WINDOW_K': '30', 'WINDOW_STRIDE':'30', 'N_TIME_MODES': '18',
    'N_VAL': '30','N_TEST':'70', 'SPLIT_MODE':'contiguous',
    'FINAL_LOSS_WEIGHT': '0', 'TIME_SUBSAMPLE': '1',
    'PYTHONUNBUFFERED':'1'
})
if SMOKE_TEST_FIRST:
    check_env = dict(ENV)
    check_env['BASELINE_SMOKE_TEST'] = '1'
    print('Quick architecture shape/gradient tests:', flush=True)
    subprocess.run([sys.executable, '-u', str(DRIVER)], env=check_env, check=True)
else:
    print('Architecture smoke test disabled by user.')

In [ ]:
#@title 5. Train/evaluate baselines, stream all logs, and save on Drive
from threading import Event, Thread

logfile=BASELINE_ROOT/'baseline_stdout.log'
print('Progress and full stdout log:', logfile, flush=True)
done=Event()

def heartbeat():
    while not done.wait(180):
        print('Baseline experiment subprocess still alive; consult last epoch output.',flush=True)

watcher=Thread(target=heartbeat,daemon=True)
watcher.start()
try:
    with logfile.open('a', encoding='utf-8') as log:
        with subprocess.Popen([sys.executable,'-u',str(DRIVER)],
                              env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            status=proc.wait()
        if status != 0:
            raise subprocess.CalledProcessError(status, [sys.executable,str(DRIVER)])
finally:
    done.set()
    watcher.join()
print('All selected baseline training/evaluation runs completed.')

In [ ]:
#@title 6. Read saved accuracy, training times and visualize numerical curves
import pandas as pd
from IPython.display import display, Image
summary=BASELINE_ROOT/'results.csv'
if summary.exists():
    result=pd.read_csv(summary)
    display(result[['model','N_train','seed','params','real_scalar_dof','full_l2_mean',
                    'final_l2_mean','training_seconds','full_test_set_inference_seconds',
                    'peak_inference_allocated_gpu_MiB']].sort_values(['N_train','model']).reset_index(drop=True))
else:
    print('No completed result rows yet.')
for figure in ('all_models_full_l2_vs_n.png','all_models_final_l2_vs_n.png'):
    path=BASELINE_ROOT/'plots'/figure
    if path.exists():
        print('Plot:',path)
        display(Image(filename=str(path), width=950))
    else:print('Not yet saved:',path)
print('All images and result CSV/JSON live at:',BASELINE_ROOT)

In [ ]:
#@title 7. Disconnect the GPU when the suite and report cells finish
if SHUTDOWN_AFTER_SUCCESS:
    from google.colab import runtime
    print('Experiment/report cells finished. Disconnecting and releasing GPU.',flush=True)
    runtime.unassign()
else:
    print('GPU kept active. Runtime → Disconnect and delete runtime when done.')